# The company words keep

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DonErnesto/the-company-words-keep/blob/main/word2vec.ipynb)

A compact implementation of **skip-gram Word2Vec with negative sampling**. The interesting part is not the tiny neural network; it is how we turn text into training examples.

For each word, skip-gram treats that word as the **center** and nearby words as its **context**. Frequent words are subsampled, genuine center–context pairs are positive examples, and randomly sampled words provide negative examples.

## Setup

Set `QUICK_RUN = True` for a short CPU-friendly demonstration. Change it to `False` to train on all 17 million words in Text8. A GPU is used automatically when available.

In [ ]:
from collections import Counter
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import torch
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
QUICK_RUN = True
print("device:", device)

## 1. Load the corpus

[Text8](https://mattmahoney.net/dc/textdata.html) is 100 MB of cleaned English Wikipedia text: large enough to learn interesting relationships, but still practical on a personal computer.

In [ ]:
def load_text8(limit=None):
    url = "https://mattmahoney.net/dc/text8.zip"
    with urlopen(url) as response, ZipFile(BytesIO(response.read())) as archive:
        words = archive.read("text8").decode().split()
    return words[:limit]

tokens = load_text8(1_000_000 if QUICK_RUN else None)
print(f"{len(tokens):,} tokens; sample:", tokens[:12])

## 2. Build the vocabulary

Words that occur only a handful of times cannot acquire reliable vectors, so we discard them and represent every retained word by an integer.

In [ ]:
def build_vocab(tokens, min_count=5):
    counts = Counter(tokens)
    vocab = [word for word, count in counts.most_common() if count >= min_count]
    word_to_id = {word: i for i, word in enumerate(vocab)}
    word_ids = torch.tensor([word_to_id[w] for w in tokens if w in word_to_id])
    frequencies = torch.tensor([counts[w] for w in vocab], dtype=torch.float)
    return vocab, word_to_id, word_ids, frequencies

vocab, word_to_id, word_ids, frequencies = build_vocab(tokens)
print(f"vocabulary: {len(vocab):,} words")

## 3. Subsample frequent words

Very common words such as *the* create enormous numbers of unsurprising pairs. Word2Vec probabilistically removes them. Rare words are kept; increasingly frequent words are increasingly likely to be dropped.

In [ ]:
def subsample(word_ids, frequencies, threshold=1e-5):
    relative_frequency = frequencies / frequencies.sum()
    keep_probability = (torch.sqrt(relative_frequency / threshold) + 1) * threshold / relative_frequency
    keep_probability.clamp_(max=1)
    return word_ids[torch.rand(len(word_ids)) < keep_probability[word_ids]]

training_words = subsample(word_ids, frequencies)
print(f"subsampling retained {len(training_words):,} of {len(word_ids):,} tokens")

## 4. Generate center–context pairs

Each center word receives a random window radius. The function yields chunks rather than constructing every pair at once, keeping memory use modest even for the full corpus.

In [ ]:
def generate_pairs(words, window=5, chunk_size=100_000):
    for start in range(0, len(words), chunk_size):
        positions = torch.arange(start, min(start + chunk_size, len(words)))
        radii = torch.randint(1, window + 1, (len(positions),))
        centers, contexts = [], []

        for offset in range(-window, window + 1):
            if offset == 0:
                continue
            valid = (positions + offset >= 0) & (positions + offset < len(words)) & (radii >= abs(offset))
            centers.append(words[positions[valid]])
            contexts.append(words[positions[valid] + offset])

        yield torch.cat(centers), torch.cat(contexts)

example_centers, example_contexts = next(generate_pairs(training_words[:20], window=2))
[(vocab[c], vocab[o]) for c, o in zip(example_centers[:8], example_contexts[:8])]

## 5. Sample negative contexts

For every genuine pair, we draw words that did not occur in that context. The original Word2Vec recipe samples from word frequencies raised to the power `0.75`: common words remain common negatives, but less overwhelmingly so.

In [ ]:
def sample_negatives(frequencies, batch_size, n_negatives=5):
    weights = frequencies.pow(0.75)
    return torch.multinomial(weights, batch_size * n_negatives, replacement=True).view(batch_size, n_negatives)

## 6. Train the shallow network

The network is just two lookup tables. A center vector should have a large dot product with its real context vector and a small dot product with sampled negative vectors. PyTorch handles the gradients; the loop below handles batching.

In [ ]:
embedding_dim = 100
batch_size = 2_048
epochs = 1 if QUICK_RUN else 3

center_embedding = torch.nn.Embedding(len(vocab), embedding_dim, sparse=True).to(device)
context_embedding = torch.nn.Embedding(len(vocab), embedding_dim, sparse=True).to(device)
with torch.no_grad():
    center_embedding.weight.uniform_(-0.5 / embedding_dim, 0.5 / embedding_dim)
    context_embedding.weight.zero_()
optimizer = torch.optim.SparseAdam(
    list(center_embedding.parameters()) + list(context_embedding.parameters()), lr=0.01
)

for epoch in range(epochs):
    running_loss, steps = 0.0, 0
    for centers, contexts in generate_pairs(training_words):
        order = torch.randperm(len(centers))
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            c, o = centers[batch].to(device), contexts[batch].to(device)
            negatives = sample_negatives(frequencies, len(batch)).to(device)

            center_vectors = center_embedding(c)
            positive_score = (center_vectors * context_embedding(o)).sum(dim=1)
            negative_score = torch.bmm(context_embedding(negatives), center_vectors.unsqueeze(2)).squeeze(2)
            loss = -(F.logsigmoid(positive_score) + F.logsigmoid(-negative_score).sum(dim=1)).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            steps += 1

    print(f"epoch {epoch + 1}: loss = {running_loss / steps:.3f}")

## 7. Explore the learned space

Cosine similarity compares directions rather than magnitudes. PyTorch computes it for the query word against the entire vocabulary. The quick run is deliberately small, so expect much better neighborhoods after training on all of Text8.

In [ ]:
def nearest_words(word, n=8):
    vectors = center_embedding.weight.detach()
    similarities = F.cosine_similarity(vectors, vectors[word_to_id[word]].unsqueeze(0), dim=1)
    best = similarities.topk(n + 1).indices.cpu().tolist()
    return [(vocab[i], round(similarities[i].item(), 3)) for i in best if vocab[i] != word][:n]

for word in ["king", "computer", "music"]:
    if word in word_to_id:
        print(word, nearest_words(word))

## What to try next

Turn off `QUICK_RUN`, change the context window, vary the number of negative samples, or compare results before and after subsampling. The learned vectors are available as `center_embedding.weight`; that first embedding table is the artifact commonly retained as the Word2Vec representation.